##Upload your **trimmed_reads.vcf**

*   This is the final output from the **job_bowtie_trimmed_RM.sh** SLURM script

In [ ]:
from google.colab import files
print("Upload your VCF (.vcf/.vcf.gz)…")
uploaded = files.upload()
list(uploaded.keys())

VCF_PATH = next((k for k in uploaded if k.lower().endswith((".vcf",".vcf.gz"))), "variants.vcf")
print("VCF:", VCF_PATH)

##Install necessary packages and download the E.coli genome in genbank format(NC_012967.1)
* The gbk format contains both the genome sequence and the gene annotations

In [ ]:
!pip -q install biopython cyvcf2 plotly
import re, numpy as np
from collections import Counter
from Bio import SeqIO
from cyvcf2 import VCF
import plotly.graph_objects as go
from Bio import Entrez

Entrez.email = "your.email@example.com"   # <-- put your email here

accession = "NC_012967.1"  # E. coli K-12 MG1655
GBK_PATH = f"{accession}.gbk"

import os
if not os.path.exists(GBK_PATH):
    handle = Entrez.efetch(db="nuccore", id=accession,
                           rettype="gbwithparts", retmode="text")
    with open(GBK_PATH, "w") as out:
        out.write(handle.read())
    handle.close()

print("Using GenBank file:", GBK_PATH)


###Parse the genbank genome file (gbk file)
* exctract the CDS postiions for all genes
* match genome to variant position scale (base-1 inclusive)

In [ ]:
def norm_chrom(s: str) -> str:
    if s is None: return ""
    s = s.strip()
    s = re.sub(r'^\s*chr', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+', '', s)
    s = re.sub(r'\.[0-9]+$', '', s)   # strip trailing .1
    return s

# Read records (some GBKs have multiple records; choose the one with most CDS or longest)
records = list(SeqIO.parse(GBK_PATH, "genbank"))
if not records:
    raise RuntimeError("No records found in GBK. Check file.")

# choose record with most CDS; tie-break by length
def cds_count(rec):
    return sum(1 for f in rec.features if f.type.lower()=="cds")
rec = sorted(records, key=lambda r: (cds_count(r), len(r)), reverse=True)[0]

chrom_key = norm_chrom(rec.id) or norm_chrom(rec.name)
chrom_len = len(rec)

# Extract gene/CDS intervals (convert to 1-based inclusive to match VCF POS)
genes, cds = [], []
for f in rec.features:
    t = f.type.lower()
    if t not in ("gene","cds"):
        continue
    s = int(f.location.start) + 1
    e = int(f.location.end)       # end is exclusive in Biopython
    strand = "+" if getattr(f.location, "strand", 1) != -1 else "-"
    if t == "gene":
        genes.append((chrom_key, s, e, strand))
    else:  # CDS
        cds.append((chrom_key, s, e, strand))

if not genes:
    raise RuntimeError("No 'gene' features found in GBK.")

# normalize (kept for symmetry with earlier code)
genes_n = [(norm_chrom(c), s, e, st) for (c,s,e,st) in genes]
cds_n   = [(norm_chrom(c), s, e, st) for (c,s,e,st) in cds]
gene_rows = [(s,e,st) for (c,s,e,st) in genes_n if c == chrom_key]
cds_rows  = [(s,e,st) for (c,s,e,st) in cds_n   if c == chrom_key]

print(f"Using record: {rec.id} | length={chrom_len:,} | genes={len(gene_rows)} | CDS={len(cds_rows)}")


###Match vcf to genome to identify variants as coding or non-coding

In [ ]:
vcf = VCF(VCF_PATH)
vars_norm = []
for r in vcf:
    if not r.ALT or r.ALT[0] is None:
        continue
    vars_norm.append((norm_chrom(r.CHROM), int(r.POS)))
chrom_vars = sorted([pos for (c,pos) in vars_norm if c == chrom_key])
print(f"Variants on {chrom_key}: {len(chrom_vars)} (total parsed: {len(vars_norm)})")

# Build CDS overlap check
cds_intervals = [(s,e) for (s,e,_) in cds_rows]
def is_coding_variant(pos):
    for s,e in cds_intervals:
        if s <= pos <= e:
            return True
    return False

coding_vars    = [p for p in chrom_vars if is_coding_variant(p)]
noncoding_vars = [p for p in chrom_vars if not is_coding_variant(p)]
print("coding:", len(coding_vars), " noncoding:", len(noncoding_vars))


##The cell below generates an interactive circos plot using plotly

## **For the Homework Assignment:**
1) **Modify the title of the Figure so it contains ONLY your <u>Name and student ID</u>**
2)   **Use the navigation buttons in the figure to "Download plot as a png" (camera icon)**
3)   **Submit that png file to complete the assignment in Canvas**



In [ ]:
def to_theta(x, L): return 2*np.pi * (x / float(L))
r_inner, r_gene = 0.62, 0.68

fig = go.Figure()

# short inner circle for tick starts
t = np.linspace(0, 2*np.pi, 600)
fig.add_trace(go.Scatter(x=r_inner*np.cos(t), y=r_inner*np.sin(t),
                         mode="lines", line=dict(width=1, color="lightgray"), showlegend=False))

# gene backbone (thicker if you want: width=3–5)
for s,e,_ in gene_rows:
    th1, th2 = to_theta(s, chrom_len), to_theta(e, chrom_len)
    tt = np.linspace(min(th1,th2), max(th1,th2), 32)
    fig.add_trace(go.Scatter(x=r_gene*np.cos(tt), y=r_gene*np.sin(tt),
                             mode="lines", line=dict(width=3, color="lightgray"), showlegend=False))

# CDS arcs (fat black)
for s,e,_ in cds_rows:
    th1, th2 = to_theta(s, chrom_len), to_theta(e, chrom_len)
    tt = np.linspace(min(th1,th2), max(th1,th2), 40)
    fig.add_trace(go.Scatter(x=r_gene*np.cos(tt), y=r_gene*np.sin(tt),
                             mode="lines", line=dict(width=12, color="black"), showlegend=False))

# Variant ticks: coding (red), noncoding (blue)
for pos in coding_vars:
    th = to_theta(pos, chrom_len)
    fig.add_trace(go.Scatter(x=[r_inner*np.cos(th), r_gene*np.cos(th)],
                             y=[r_inner*np.sin(th), r_gene*np.sin(th)],
                             mode="lines", line=dict(width=2, color="red"), showlegend=False))
for pos in noncoding_vars:
    th = to_theta(pos, chrom_len)
    fig.add_trace(go.Scatter(x=[r_inner*np.cos(th), r_gene*np.cos(th)],
                             y=[r_inner*np.sin(th), r_gene*np.sin(th)],
                             mode="lines", line=dict(width=2, color="blue"), showlegend=False))

# tidy layout
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
title_txt = f"{chrom_key}: Coding (red) vs noncoding (blue) variants<br>Genes outer ring, CDS (black))"
fig.update_layout(
    width=760, height=760, dragmode="pan",
    margin=dict(l=10,r=10,t=90,b=10),
    title=dict(text=title_txt, x=0.5, xanchor="center", y=0.95, yanchor="top"),
    plot_bgcolor="white", paper_bgcolor="white"
)
fig.show()
